# TF-IDF + XGBoost Pipeline (Single Config)
Simple pipeline for fake news detection with default hyperparameters

In [1]:
# ============================================================================
# IMPORTS
# ============================================================================
from __future__ import annotations

import json
import re
import shutil
from pathlib import Path
from typing import Dict, List, Sequence, Tuple
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse import csr_matrix, hstack
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

import torch
from transformers import RobertaTokenizer, RobertaModel
from tqdm import tqdm

sns.set_theme(style="whitegrid")
print("✅ All imports successful!")

✅ All imports successful!


In [2]:
# ============================================================================
# CONSTANTS & SETUP
# ============================================================================
RANDOM_STATE = 42

# Adjust paths based on your working directory
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "full_data.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
PLOTS_DIR = RESULTS_DIR / "plots"
CM_DIR = RESULTS_DIR / "confusion_matrix"
MODELS_DIR = RESULTS_DIR / "models"

ENGINEERED_FEATURE_NAMES = [
    "content_char_len",
    "content_word_len",
    "exclamation_count",
    "question_count",
    "uppercase_ratio",
]

def ensure_output_dirs() -> None:
    for directory in [RESULTS_DIR, PLOTS_DIR, CM_DIR, MODELS_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

ensure_output_dirs()
print(f"Project root: {PROJECT_ROOT}")
print(f"Output directories created ✅")

Project root: d:\HK8\hybrid-fake-news-detector\src
Output directories created ✅


In [ ]:
# ============================================================================
# DATA PREPROCESSING
# ============================================================================
def normalize_schema(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename_map = {}
    if "text" in df.columns and "content" not in df.columns:
        rename_map["text"] = "content"
    if "tweet_id" in df.columns and "id" not in df.columns:
        rename_map["tweet_id"] = "id"
    if rename_map:
        df = df.rename(columns=rename_map)
    if "content" not in df.columns:
        raise ValueError("Dataset must contain a text column named 'content' or 'text'.")
    if "label" not in df.columns:
        raise ValueError("Dataset must contain a 'label' column.")
    return df


def canonical_label(value: object) -> str:
    text = str(value).strip().lower()
    text = re.sub(r"[\s_\-]+", "", text)
    return text


def encode_labels(labels: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(labels):
        numeric = pd.to_numeric(labels, errors="coerce")
        valid = set(numeric.dropna().unique().tolist())
        if valid.issubset({0, 1}):
            return numeric.astype(int)

    mapping = {
        "nonrumor": 0,
        "truth": 0,
        "true": 0,
        "real": 0,
        "legit": 0,
        "legitimate": 0,
        "false": 1,
        "rumor": 1,
        "fake": 1,
        "unverified": 1,
    }
    encoded = labels.map(lambda x: mapping.get(canonical_label(x)))
    if encoded.isna().any():
        invalid = sorted({str(v) for v in labels[encoded.isna()].unique().tolist()})
        raise ValueError(
            f"Unsupported labels found. Expected values similar to true/non-rumor/false/unverified. "
            f"Invalid values: {invalid}"
        )
    return encoded.astype(int)


def clean_text(text: object) -> str:
    if pd.isna(text):
        return ""
    value = str(text).lower()
    value = re.sub(r"http\S+|www\S+|https\S+", " ", value)
    value = re.sub(r"@[A-Za-z0-9_]+", " ", value)
    value = re.sub(r"#[A-Za-z0-9_]+", " ", value)
    value = re.sub(r"[^a-z\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def extract_engineered_features(texts: Sequence[object]) -> np.ndarray:
    rows: List[List[float]] = []
    for raw_text in texts:
        value = "" if pd.isna(raw_text) else str(raw_text)
        word_count = len(value.split())
        upper_alpha = sum(1 for ch in value if ch.isalpha() and ch.isupper())
        alpha_count = sum(1 for ch in value if ch.isalpha())
        uppercase_ratio = (upper_alpha / alpha_count) if alpha_count else 0.0
        rows.append(
            [
                float(len(value)),
                float(word_count),
                float(value.count("!")),
                float(value.count("?")),
                float(uppercase_ratio),
            ]
        )
    return np.asarray(rows, dtype=np.float32)

print("✅ Data preprocessing functions defined")

✅ Data preprocessing functions defined


In [ ]:
# ============================================================================
# ROBERTA SETUP
# ============================================================================
print("Setting up RoBERTa model...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
roberta_model = RobertaModel.from_pretrained("roberta-base")
roberta_model.to(device)
roberta_model.eval()

print("✅ RoBERTa model loaded successfully!")


def get_roberta_embedding(text):
    """Get [CLS] token embedding from RoBERTa (better for classification)."""
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = roberta_model(**inputs)
    # Use [CLS] token (index 0) instead of mean pooling
    return outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()


def extract_embeddings(texts):
    """Extract embeddings for a list of texts."""
    return np.vstack([get_roberta_embedding(text) for text in tqdm(texts, desc="Extracting RoBERTa embeddings")])

Setting up RoBERTa model...
Using device: cpu


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ RoBERTa model loaded successfully!


In [ ]:
# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
class TextFeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        max_features: int = 768,
        min_df: int = 2,
        max_df: float = 0.9,
        ngram_range: Tuple[int, int] = (1, 2),
    ) -> None:
        self.max_features = max_features
        self.min_df = min_df
        self.max_df = max_df
        self.ngram_range = ngram_range
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            min_df=min_df,
            max_df=max_df,
            ngram_range=ngram_range,
            stop_words="english",
        )

    def fit(self, X: Sequence[object], y: Sequence[int] | None = None):
        raw_texts = ["" if pd.isna(text) else str(text) for text in X]
        cleaned_texts = [clean_text(text) for text in raw_texts]
        self.vectorizer.fit(cleaned_texts)
        return self

    def transform(self, X: Sequence[object]):
        raw_texts = ["" if pd.isna(text) else str(text) for text in X]
        cleaned_texts = [clean_text(text) for text in raw_texts]
        tfidf_matrix = self.vectorizer.transform(cleaned_texts)
        engineered = csr_matrix(extract_engineered_features(raw_texts))
        return hstack([tfidf_matrix, engineered], format="csr")

    def get_feature_names_out(self) -> np.ndarray:
        tfidf_names = self.vectorizer.get_feature_names_out()
        return np.concatenate([tfidf_names, np.asarray(ENGINEERED_FEATURE_NAMES, dtype=object)])

print("✅ TextFeatureBuilder class defined")

✅ TextFeatureBuilder class defined


In [6]:
# ============================================================================
# LOAD DATASET
# ============================================================================
print("\n" + "="*80)
print("LOADING DATA")
print("="*80)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} total records")

df = normalize_schema(df)
df = df[["content", "label"]].copy()
df["content"] = df["content"].fillna("").astype(str)
df["label"] = encode_labels(df["label"])

print(f"Label distribution:")
print(df["label"].value_counts().sort_index())


LOADING DATA
Loaded 2139 total records
Label distribution:
label
0    1158
1     981
Name: count, dtype: int64


In [7]:
# ============================================================================
# TRAIN-TEST SPLIT (DEFAULT CONFIG: 80/20)
# ============================================================================
print("\n" + "="*80)
print("TRAIN-TEST SPLIT (80/20)")
print("="*80)

X = df["content"]
y = df["label"].to_numpy(dtype=int)

test_size = 0.20  # 80/20 split (keep 80% for train)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
)

print(f"Train set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Train label distribution:\n{pd.Series(y_train).value_counts().sort_index()}")
print(f"Test label distribution:\n{pd.Series(y_test).value_counts().sort_index()}")


TRAIN-TEST SPLIT (80/20)
Train set size: 1711
Test set size: 428
Train label distribution:
0    926
1    785
Name: count, dtype: int64
Test label distribution:
0    232
1    196
Name: count, dtype: int64


In [8]:
# ============================================================================
# EXTRACT FEATURES (TF-IDF + RoBERTa + ENGINEERED)
# ============================================================================
print("\n" + "="*80)
print("FEATURE EXTRACTION")
print("="*80)

# Initialize TextFeatureBuilder
feature_builder = TextFeatureBuilder(
    max_features=1536,
    min_df=4,
    max_df=0.9,
    ngram_range=(1, 2),
)

# Fit the feature builder on training data
print("\n→ Fitting TextFeatureBuilder (TRAIN)...")
feature_builder.fit(X_train)

# Extract TF-IDF features from both train and test
print("→ Extracting TF-IDF features (TRAIN)...")
X_train_tfidf = feature_builder.transform(X_train)
print(f"  TF-IDF shape: {X_train_tfidf.shape}")

print("→ Extracting TF-IDF features (TEST)...")
X_test_tfidf = feature_builder.transform(X_test)
print(f"  TF-IDF shape: {X_test_tfidf.shape}")

# Convert sparse matrices to dense for combination with other features
X_train_tfidf = X_train_tfidf.toarray()
X_test_tfidf = X_test_tfidf.toarray()

# Extract RoBERTa embeddings
print("\n→ Extracting RoBERTa embeddings (TRAIN)...")
X_train_roberta = extract_embeddings(X_train.values)
print(f"  RoBERTa shape: {X_train_roberta.shape}")

print("→ Extracting RoBERTa embeddings (TEST)...")
X_test_roberta = extract_embeddings(X_test.values)
print(f"  RoBERTa shape: {X_test_roberta.shape}")

# Scale RoBERTa embeddings
print("\n→ Scaling RoBERTa embeddings...")
scaler = StandardScaler()
scaler.fit(X_train_roberta)
X_train_roberta_scaled = scaler.transform(X_train_roberta)
X_test_roberta_scaled = scaler.transform(X_test_roberta)

# Extract engineered features
print("→ Extracting engineered features (TRAIN)...")
X_train_engineered = extract_engineered_features(X_train)
print(f"  Engineered shape: {X_train_engineered.shape}")

print("→ Extracting engineered features (TEST)...")
X_test_engineered = extract_engineered_features(X_test)
print(f"  Engineered shape: {X_test_engineered.shape}")

# Combine all features: TF-IDF + RoBERTa + Engineered
print("\n→ Combining all features...")
X_train_features = np.hstack([X_train_tfidf, X_train_roberta_scaled, X_train_engineered])
X_test_features = np.hstack([X_test_tfidf, X_test_roberta_scaled, X_test_engineered])

print(f"\n✅ FINAL FEATURE SHAPES:")
print(f"  Train: {X_train_features.shape} (TF-IDF + RoBERTa + Engineered)")
print(f"  Test:  {X_test_features.shape} (TF-IDF + RoBERTa + Engineered)")

# Store feature builder's vectorizer for later use
tfidf_vec = feature_builder.vectorizer


FEATURE EXTRACTION

→ Fitting TextFeatureBuilder (TRAIN)...
→ Extracting TF-IDF features (TRAIN)...
  TF-IDF shape: (1711, 1283)
→ Extracting TF-IDF features (TEST)...
  TF-IDF shape: (428, 1283)

→ Extracting RoBERTa embeddings (TRAIN)...


Extracting RoBERTa embeddings: 100%|██████████| 1711/1711 [02:19<00:00, 12.29it/s]


  RoBERTa shape: (1711, 768)
→ Extracting RoBERTa embeddings (TEST)...


Extracting RoBERTa embeddings: 100%|██████████| 428/428 [00:29<00:00, 14.65it/s]

  RoBERTa shape: (428, 768)

→ Scaling RoBERTa embeddings...
→ Extracting engineered features (TRAIN)...
  Engineered shape: (1711, 5)
→ Extracting engineered features (TEST)...
  Engineered shape: (428, 5)

→ Combining all features...

✅ FINAL FEATURE SHAPES:
  Train: (1711, 2056) (TF-IDF + RoBERTa + Engineered)
  Test:  (428, 2056) (TF-IDF + RoBERTa + Engineered)


In [9]:
# ============================================================================
# TRAIN XGBOOST (DEFAULT CONFIG)
# ============================================================================
print("\n" + "="*80)
print("TRAINING XGBOOST")
print("="*80)

config = {
    "n_estimators": 300,
    "max_depth": 5,
    "learning_rate": 0.07,
}

print(f"\nDefault Configuration:")
print(f"  n_estimators: {config['n_estimators']}")
print(f"  max_depth: {config['max_depth']}")
print(f"  learning_rate: {config['learning_rate']}")

# Compute class weights
scale_pos_weight = float(np.sum(y_train == 0)) / float(np.sum(y_train == 1))
print(f"\nClass weight (negative/positive): {scale_pos_weight:.4f}")

# Split training data into train and validation for loss tracking
X_train_fit, X_val, y_train_fit, y_val = train_test_split(
    X_train_features, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)

# Train model with validation set for loss and accuracy tracking
print("\nTraining...")
model = XGBClassifier(
    n_estimators=config["n_estimators"],
    max_depth=config["max_depth"],
    learning_rate=config["learning_rate"],
    eval_metric=["logloss", "error"],
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    importance_type="gain",
    use_label_encoder=False,
)

# Train with eval_set to track loss and accuracy
eval_set = [(X_train_fit, y_train_fit), (X_val, y_val)]
model.fit(
    X_train_fit, 
    y_train_fit,
    eval_set=eval_set,
    verbose=False
)

# Store loss and accuracy history for visualization
results = model.evals_result()
train_loss = results['validation_0']['logloss']
val_loss = results['validation_1']['logloss']
train_error = results['validation_0']['error']
val_error = results['validation_1']['error']

# Convert error to accuracy (accuracy = 1 - error)
train_accuracy = [1 - e for e in train_error]
val_accuracy = [1 - e for e in val_error]

# Retrain on full training set for final model
print("\n→ Retraining on full training set...")
model = XGBClassifier(
    n_estimators=config["n_estimators"],
    max_depth=config["max_depth"],
    learning_rate=config["learning_rate"],
    eval_metric=["logloss", "error"],
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    importance_type="gain",
    use_label_encoder=False,
)
model.fit(X_train_features, y_train, verbose=False)

print("✅ Training complete!")


TRAINING XGBOOST

Default Configuration:
  n_estimators: 300
  max_depth: 5
  learning_rate: 0.07

Class weight (negative/positive): 1.1796

Training...


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [09:20:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



→ Retraining on full training set...


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [09:20:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Training complete!


In [10]:
# ============================================================================
# EVALUATE ON TEST SET
# ============================================================================
print("\n" + "="*80)
print("EVALUATION")
print("="*80)

y_prob = model.predict_proba(X_test_features)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

# Compute metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred, zero_division=0)
test_recall = recall_score(y_test, y_pred, zero_division=0)
test_f1 = f1_score(y_test, y_pred, zero_division=0)
test_roc_auc = roc_auc_score(y_test, y_prob)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]

print(f"\n📊 TEST SET METRICS:")
print(f"  Accuracy:  {test_accuracy:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1 Score:  {test_f1:.4f}")
print(f"  ROC-AUC:   {test_roc_auc:.4f}")

print(f"\n🎯 CONFUSION MATRIX:")
print(f"     Pred 0  Pred 1")
print(f"Act 0  {tn:5d}  {fp:5d}")
print(f"Act 1  {fn:5d}  {tp:5d}")

print(f"\n📋 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, target_names=["Truth", "Rumor"]))


EVALUATION

📊 TEST SET METRICS:
  Accuracy:  0.8037
  Precision: 0.7917
  Recall:    0.7755
  F1 Score:  0.7835
  ROC-AUC:   0.8731

🎯 CONFUSION MATRIX:
     Pred 0  Pred 1
Act 0    192     40
Act 1     44    152

📋 CLASSIFICATION REPORT:
              precision    recall  f1-score   support

       Truth       0.81      0.83      0.82       232
       Rumor       0.79      0.78      0.78       196

    accuracy                           0.80       428
   macro avg       0.80      0.80      0.80       428
weighted avg       0.80      0.80      0.80       428



In [11]:
# ============================================================================
# VISUALIZATIONS
# ============================================================================
print("\n" + "="*80)
print("GENERATING PLOTS")
print("="*80)

# Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Truth", "Rumor"],
    yticklabels=["Truth", "Rumor"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig(CM_DIR / "confusion_matrix.png", dpi=200)
plt.close()
print(f"✅ Saved: confusion_matrix.png")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2.5, label=f"ROC AUC = {test_roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.5, label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "roc_curve.png", dpi=200)
plt.close()
print(f"✅ Saved: roc_curve.png")


GENERATING PLOTS
✅ Saved: confusion_matrix.png
✅ Saved: roc_curve.png


In [12]:
# Loss Curve
plt.figure(figsize=(10, 6))
epochs = range(1, len(train_loss) + 1)
plt.plot(epochs, train_loss, linewidth=2, label="Train Loss (LogLoss)", marker='o', markersize=3, alpha=0.7)
plt.plot(epochs, val_loss, linewidth=2, label="Validation Loss (LogLoss)", marker='s', markersize=3, alpha=0.7)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "loss_curve.png", dpi=200)
plt.close()
print(f"✅ Saved: loss_curve.png")

✅ Saved: loss_curve.png


In [13]:
# ============================================================================
# FEATURE IMPORTANCE
# ============================================================================
print("\nExtracting feature importance...")

# Get feature names from TextFeatureBuilder
all_feature_names_from_builder = feature_builder.get_feature_names_out()

# Separate TF-IDF feature names and engineered feature names
n_tfidf_features = len(all_feature_names_from_builder) - len(ENGINEERED_FEATURE_NAMES)
tfidf_feature_names = all_feature_names_from_builder[:n_tfidf_features]
engineered_feature_names = all_feature_names_from_builder[n_tfidf_features:]

# Generate RoBERTa feature names
roberta_feature_names = np.array([f"RoBERTa_dim_{i}" for i in range(X_train_roberta_scaled.shape[1])])

# Combine all feature names
all_feature_names = np.concatenate([
    tfidf_feature_names,
    roberta_feature_names,
    engineered_feature_names
])

importances = np.asarray(model.feature_importances_, dtype=float)

top_k = min(20, importances.size)
top_indices = np.argsort(importances)[::-1][:top_k]
top_features = all_feature_names[top_indices]
top_values = importances[top_indices]

# Plot Feature Importance
plt.figure(figsize=(10, 7))
sns.barplot(x=top_values, y=top_features, palette="viridis")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 Feature Importances (TF-IDF + RoBERTa + Engineered)")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "feature_importance.png", dpi=200)
plt.close()
print(f"✅ Saved: feature_importance.png")

# Print top features
print(f"\nTop 20 Features by Importance:")
for rank, (feat, val) in enumerate(zip(top_features, top_values), 1):
    print(f"  {rank:2d}. {feat:40s} → {val:.6f}")


Extracting feature importance...


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_25428\3082749400.py:33: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_values, y=top_features, palette="viridis")


✅ Saved: feature_importance.png

Top 20 Features by Importance:
   1. walker                                   → 0.009350
   2. transgender                              → 0.009036
   3. white                                    → 0.007520
   4. RoBERTa_dim_648                          → 0.007258
   5. RoBERTa_dim_609                          → 0.006287
   6. RoBERTa_dim_426                          → 0.006030
   7. paul                                     → 0.005942
   8. RoBERTa_dim_246                          → 0.005753
   9. RoBERTa_dim_10                           → 0.005640
  10. RoBERTa_dim_82                           → 0.005514
  11. RoBERTa_dim_141                          → 0.005182
  12. RoBERTa_dim_494                          → 0.005026
  13. rainbow                                  → 0.004817
  14. RoBERTa_dim_220                          → 0.004645
  15. RoBERTa_dim_412                          → 0.004471
  16. RoBERTa_dim_111                          → 0.004447
  17. Ro

In [14]:
# Accuracy Curve
plt.figure(figsize=(10, 6))
epochs = range(1, len(train_accuracy) + 1)
plt.plot(epochs, train_accuracy, linewidth=2, label="Train Accuracy", marker='o', markersize=3, alpha=0.7)
plt.plot(epochs, val_accuracy, linewidth=2, label="Validation Accuracy", marker='s', markersize=3, alpha=0.7)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.ylim([0, 1])
plt.tight_layout()
plt.savefig(PLOTS_DIR / "accuracy_curve.png", dpi=200)
plt.close()
print(f"✅ Saved: accuracy_curve.png")

✅ Saved: accuracy_curve.png


In [15]:
# ============================================================================
# SAVE RESULTS & MODELS
# ============================================================================
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

# Results JSON
results = {
    "timestamp": datetime.now().isoformat(),
    "config": config,
    "split_ratio": "80/20",
    "features": "TF-IDF (768) + RoBERTa (768) + Engineered (5)",
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "metrics": {
        "accuracy": float(test_accuracy),
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1_score": float(test_f1),
        "roc_auc": float(test_roc_auc),
    },
    "confusion_matrix": {
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
    },
}

with open(RESULTS_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("✅ Saved: results.json")

# Save model
joblib.dump(model, MODELS_DIR / "xgboost_model.joblib")
print("✅ Saved: xgboost_model.joblib")

# Save TF-IDF vectorizer
joblib.dump(tfidf_vec, MODELS_DIR / "tfidf_vectorizer.joblib")
print("✅ Saved: tfidf_vectorizer.joblib")

# Save scaler
joblib.dump(scaler, MODELS_DIR / "roberta_scaler.joblib")
print("✅ Saved: roberta_scaler.joblib")

print("\n" + "="*80)
print("✅ PIPELINE COMPLETE!")
print("="*80)
print(f"\nResults saved to: {RESULTS_DIR}")
print(f"Plots saved to: {PLOTS_DIR}")
print(f"Models saved to: {MODELS_DIR}")


SAVING RESULTS
✅ Saved: results.json
✅ Saved: xgboost_model.joblib
✅ Saved: tfidf_vectorizer.joblib
✅ Saved: roberta_scaler.joblib

✅ PIPELINE COMPLETE!

Results saved to: d:\HK8\hybrid-fake-news-detector\src\results
Plots saved to: d:\HK8\hybrid-fake-news-detector\src\results\plots
Models saved to: d:\HK8\hybrid-fake-news-detector\src\results\models


In [16]:
# ============================================================================
# SUMMARY
# ============================================================================
print("\n📋 FINAL SUMMARY")
print("="*80)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"ROC-AUC Score: {test_roc_auc:.4f}")
print("="*80)


📋 FINAL SUMMARY
Test Accuracy: 0.8037
Test F1 Score: 0.7835
ROC-AUC Score: 0.8731
